# BCS 100 Pro — Google Colab Edition

This notebook launches the **BCS 100 Pro** web application directly from its GitHub repository.

### What this notebook does
- Downloads or updates the latest project from GitHub
- Checks the required application files
- Runs the JavaScript test suite when Node.js is available
- Starts a Python static web server
- Opens the app through Google Colab's secure proxy

> **Important:** A Colab runtime is temporary. The app remains available only while this notebook session is running. Browser progress is stored locally on the device. Use GitHub Pages for permanent hosting.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

REPOSITORY_URL = "https://github.com/cseteesta-bit/AMR.git"
PROJECT_DIR = Path("/content/AMR")
PORT = 4173

if PROJECT_DIR.exists():
    print("Updating the existing project copy...")
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "reset", "--hard", "HEAD"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    print("Cloning BCS 100 Pro...")
    subprocess.run(
        ["git", "clone", "--depth", "1", REPOSITORY_URL, str(PROJECT_DIR)],
        check=True,
    )

os.chdir(PROJECT_DIR)
print(f"Project ready: {PROJECT_DIR}")


In [ ]:
from pathlib import Path
import shutil
import subprocess

required_files = [
    "index.html",
    "manifest.json",
    "sw.js",
    "css/styles.css",
    "css/enhancements.css",
    "js/data.js",
    "js/quiz-core.js",
    "js/app-base.js",
    "js/app-views.js",
    "js/app-quiz.js",
    "js/app-init.js",
]

missing = [name for name in required_files if not (PROJECT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

print("Required-file check passed.")

if shutil.which("node") and shutil.which("npm"):
    print("\nRunning the project test suite...")
    subprocess.run(["npm", "test"], cwd=PROJECT_DIR, check=True)
else:
    print("\nNode.js/npm is unavailable in this runtime.")
    print("The app can still run, but JavaScript automated tests were skipped.")


In [ ]:
import socket
import subprocess
import sys
import time
from pathlib import Path

def port_is_open(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        return sock.connect_ex(("127.0.0.1", port)) == 0

if "bcs_server" in globals() and bcs_server.poll() is None:
    bcs_server.terminate()
    try:
        bcs_server.wait(timeout=5)
    except subprocess.TimeoutExpired:
        bcs_server.kill()

bcs_server = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "http.server",
        str(PORT),
        "--bind",
        "0.0.0.0",
    ],
    cwd=PROJECT_DIR,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

for _ in range(20):
    if port_is_open(PORT):
        break
    time.sleep(0.25)
else:
    raise RuntimeError("The local app server did not start.")

print(f"BCS 100 Pro server is running on port {PORT}.")


In [ ]:
from google.colab import output
from IPython.display import HTML, IFrame, display

app_url = output.eval_js(f"google.colab.kernel.proxyPort({PORT})")

display(
    HTML(
        f'''
        <div style="padding:14px;border:1px solid #ddd;border-radius:12px;margin:8px 0 16px">
          <b>BCS 100 Pro is running.</b><br>
          <a href="{app_url}" target="_blank" rel="noopener noreferrer">
            Open the app in a new tab
          </a>
        </div>
        '''
    )
)

display(IFrame(src=app_url, width="100%", height=850))


## Refresh to the latest GitHub version

Run the next cell after new changes are pushed to the GitHub repository. It updates the files and restarts the local Colab server.


In [ ]:
import subprocess
import sys
import time

subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "reset", "--hard", "HEAD"],
    check=True,
)
subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"],
    check=True,
)

if "bcs_server" in globals() and bcs_server.poll() is None:
    bcs_server.terminate()
    try:
        bcs_server.wait(timeout=5)
    except subprocess.TimeoutExpired:
        bcs_server.kill()

bcs_server = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "http.server",
        str(PORT),
        "--bind",
        "0.0.0.0",
    ],
    cwd=PROJECT_DIR,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

time.sleep(1)
print("Project updated and server restarted.")
print("Run the previous 'Open the app' cell again.")


## Stop the Colab server

Run the final cell when you are finished.


In [ ]:
if "bcs_server" in globals() and bcs_server.poll() is None:
    bcs_server.terminate()
    try:
        bcs_server.wait(timeout=5)
    except subprocess.TimeoutExpired:
        bcs_server.kill()
    print("BCS 100 Pro server stopped.")
else:
    print("No active BCS 100 Pro server was found.")
